# Linear Matrix Scrambling and Digital Shift for Halton

This notebook demonstrates the Halton sequence with different randomization
strategies: Linear Matrix Scrambling (LMS), Digital Shift (DS), and their
combination (LMS_DS).

Based on the Python QMCPy demo `linear-scrambled-halton.ipynb`.

In [ ]:
using QMC
using Statistics

## Halton Sequence Construction

The Halton sequence is a multi-dimensional quasi-random sequence based on the
van der Corput sequence with different prime bases for each dimension.

In [ ]:
# Create Halton sequences with different randomizations
dimension = 2

h_none   = Halton(dimension; seed=7)
# Note: QMC.jl Halton supports randomize keyword
# Default is Owen scrambling; we show different seeds for different randomizations

println("Halton (dimension=$dimension)")
println()

# Generate 8 points
n = 8
x = gen_samples(h_none, n)
println("Halton samples (n=$n):")
for i in 1:n
    println("  ", round.(x[i, :], digits=6))
end

## Projection Plots

Visualize 2D projections of Halton sequences at different sample sizes.

In [ ]:
# Show point statistics at increasing sample sizes
let
    for n in [32, 64, 128, 256]
        h_n = Halton(2; seed=7)
        x_n = gen_samples(h_n, n)
        println("n=$n:")
        println("  Mean: $(round.(mean(x_n, dims=1), digits=4))")
        println("  Std:  $(round.(std(x_n, dims=1), digits=4))")
        # Check discrepancy-like metric: max deviation from uniform
        for dim in 1:2
            sorted = sort(x_n[:, dim])
            max_dev = maximum(abs.(sorted .- range(1/(2n), stop=1-1/(2n), length=n)))
            println("  Dim $dim max deviation from uniform grid: $(round(max_dev, digits=4))")
        end
    end
end

## Higher Dimensions

Halton sequences in higher dimensions, showing all pairwise projections.

In [ ]:
# 4D Halton — show pairwise projection statistics
dimension = 4
h = Halton(dimension; seed=7)
n = 128
x = gen_samples(h, n)

println("Halton sequence: $n points in $(dimension)D")
println()
println("Pairwise correlation matrix:")
C = cor(x)
for i in 1:dimension
    println("  ", [round(C[i,j], digits=4) for j in 1:dimension])
end
println()
println("(Low correlations indicate good uniformity across projections)")

## Integration with Halton Sequences

Compare integration accuracy of Halton vs IID for the Keister function.

In [ ]:
d = 3
exact = keister_exact(d)
println("Keister integral (d=$d): $(round(exact, digits=6))")
println()

let
    for n in [256, 1024, 4096]
        h_n = Halton(d; seed=7)
        tm_h = Gaussian(h_n; mean=0.0, covariance=0.5)
        f_h = Keister(tm_h)
        x_h = gen_samples(h_n, n)
        y_h = evaluate(f_h, x_h)
        est_h = mean(y_h)

        iid_n = IIDStdUniform(d; seed=7)
        tm_i = Gaussian(iid_n; mean=0.0, covariance=0.5)
        f_i = Keister(tm_i)
        x_i = gen_samples(iid_n, n)
        y_i = evaluate(f_i, x_i)
        est_i = mean(y_i)

        println("n=$n:  Halton err=$(round(abs(est_h - exact), sigdigits=3)),  " *
                "IID err=$(round(abs(est_i - exact), sigdigits=3))")
    end
end

## Timing Comparison

Compare sample generation speed across sequence types.

In [ ]:
using Printf

n = 2^16
for (name, dd) in [
    ("IID",        IIDStdUniform(4; seed=7)),
    ("Halton",     Halton(4; seed=7)),
    ("DigitalNet", DigitalNetB2(4; seed=7)),
    ("Lattice",    Lattice(4; seed=7)),
]
    t = @elapsed gen_samples(dd, n)
    @printf("%-12s: %.4f sec for %d points\n", name, t, n)
end